In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/cleaned_retail_data.csv",
    parse_dates=["InvoiceDate"]
)

C:\Users\vicky\AppData\Local\Temp\ipykernel_680\1129216024.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df["Date"] = df["InvoiceDate"].dt.date

In [4]:
sales_daily = (
    df.groupby(["Date", "StockCode"])
      .agg(
          units_sold=("Quantity", "sum"),
          revenue=("Revenue", "sum"),
          unit_price=("UnitPrice", "mean")
      )
      .reset_index()
)

sales_daily.head()

,Date,StockCode,units_sold,revenue,unit_price
0,2009-12-01,10002,12,10.2,0.850
1,2009-12-01,10120,60,12.6,0.210
2,2009-12-01,10123C,3,3.9,1.300
3,2009-12-01,10123G,2,3.4,1.700
4,2009-12-01,10125,5,5.1,1.275


In [5]:
sales_daily.to_csv(
    "../data/processed/sales_daily.csv",
    index=False
)

In [6]:
sku_master = (
    df.groupby("StockCode")
      .agg(
          description=("Description", "first"),
          list_price=("UnitPrice", "mean")
      )
      .reset_index()
)

sku_master.head()

,StockCode,description,list_price
0,10002,INFLATABLE POLITICAL GLOBE,0.979155
1,10002R,ROBOT PENCIL SHARPNER,5.133333
2,10080,GROOVY CACTUS INFLATABLE,0.505000
3,10109,BENDY COLOUR PENCILS,0.420000
4,10120,DOGGY RUBBER,0.240137


In [7]:
sku_master["category"] = "General"

sku_master["subcategory"] = "Retail"

sku_master["unit_cost"] = sku_master["list_price"] * 0.70

In [8]:
sku_master.to_csv(
    "../data/processed/sku_master.csv",
    index=False
)

In [9]:
calendar = pd.DataFrame()

calendar["Date"] = pd.date_range(
    df["InvoiceDate"].min().date(),
    df["InvoiceDate"].max().date()
)

calendar["Year"] = calendar["Date"].dt.year
calendar["Month"] = calendar["Date"].dt.month
calendar["Week"] = calendar["Date"].dt.isocalendar().week
calendar["Quarter"] = calendar["Date"].dt.quarter
calendar["Day"] = calendar["Date"].dt.day
calendar["DayName"] = calendar["Date"].dt.day_name()

calendar.head()

,Date,Year,Month,Week,Quarter,Day,DayName
0,2009-12-01,2009,12,49,4,1,Tuesday
1,2009-12-02,2009,12,49,4,2,Wednesday
2,2009-12-03,2009,12,49,4,3,Thursday
3,2009-12-04,2009,12,49,4,4,Friday
4,2009-12-05,2009,12,49,4,5,Saturday


In [10]:
def season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    else:
        return "Autumn"


calendar["Season"] = calendar["Month"].apply(season)

In [11]:
calendar.to_csv(
    "../data/processed/calendar.csv",
    index=False
)

In [12]:
inventory = (
    sales_daily.groupby("StockCode")
    .agg(
        avg_daily_sales=("units_sold", "mean")
    )
    .reset_index()
)

In [13]:
np.random.seed(42)

inventory["on_hand_units"] = (
    inventory["avg_daily_sales"] * np.random.randint(10,30,len(inventory))
).astype(int)

inventory["on_order_units"] = np.random.randint(
    0,
    100,
    len(inventory)
)

inventory["lead_time_days"] = np.random.randint(
    5,
    20,
    len(inventory)
)

inventory["reorder_point"] = (
    inventory["avg_daily_sales"] *
    inventory["lead_time_days"]
).astype(int)

In [14]:
inventory.to_csv(
    "../data/processed/inventory_snapshots.csv",
    index=False
)